In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
import math


@dataclass(frozen=True)
class BatteryCell:
    """Simple cell descriptor (like SAM's defaults in your screenshot)."""
    v_nom: float = 3.6          # V
    capacity_ah: float = 2.25   # Ah


@dataclass(frozen=True)
class BatteryBankSizing:
    """Computed bank configuration + key electrical properties."""
    desired_power_kw: float
    desired_energy_kwh: float
    desired_bank_voltage_v: float

    cells_in_series: int
    strings_in_parallel: int
    total_cells: int

    nominal_bank_voltage_v: float
    nominal_bank_capacity_kwh: float
    nominal_bank_capacity_ah: float

    max_discharge_current_a: float
    max_charge_current_a: float
    max_c_rate_discharge_per_h: float
    max_c_rate_charge_per_h: float
    time_at_max_power_h: float


def size_battery_bank(
    desired_power_kw: float = 300.0,
    desired_energy_kwh: float = 1200.0,
    desired_bank_voltage_v: float = 500.0,
    cell: BatteryCell = BatteryCell(),
    max_c_rate_charge_per_h: float | None = None,     # if None -> implied by power/energy
    max_c_rate_discharge_per_h: float | None = None,  # if None -> implied by power/energy
) -> BatteryBankSizing:
    """
    Replicates the sizing logic shown in SAM Battery Bank Sizing:
      1) pick cells_in_series from desired DC voltage
      2) pick strings_in_parallel from desired energy
      3) compute currents, C-rates, time at max power
    """

    # 1) Series count to reach desired bank voltage (SAM uses integer cells)
    n_series = int(math.ceil(desired_bank_voltage_v / cell.v_nom))
    v_bank_nom = n_series * cell.v_nom

    # 2) Parallel strings to reach desired energy
    # Energy per string (kWh) = V_string * Ah_cell / 1000
    e_string_kwh = (v_bank_nom * cell.capacity_ah) / 1000.0
    n_parallel = int(math.ceil(desired_energy_kwh / e_string_kwh))

    total_cells = n_series * n_parallel

    # Resulting nominal capacity
    bank_capacity_ah = n_parallel * cell.capacity_ah
    bank_capacity_kwh = (v_bank_nom * bank_capacity_ah) / 1000.0

    # Currents at desired power (DC)
    p_w = desired_power_kw * 1000.0
    i_at_p = p_w / v_bank_nom  # A

    # Implied time at max power
    time_h = bank_capacity_kwh / desired_power_kw if desired_power_kw > 0 else math.inf

    # If C-rate limits not given, infer from the power/energy ratio (e.g., 300/1200 = 0.25C)
    implied_c_rate = desired_power_kw / bank_capacity_kwh if bank_capacity_kwh > 0 else 0.0
    c_rate_dis = implied_c_rate if max_c_rate_discharge_per_h is None else max_c_rate_discharge_per_h
    c_rate_chg = implied_c_rate if max_c_rate_charge_per_h is None else max_c_rate_charge_per_h

    # Max current from C-rate: Imax = C_rate * Ah_bank
    i_max_dis = c_rate_dis * bank_capacity_ah
    i_max_chg = c_rate_chg * bank_capacity_ah

    return BatteryBankSizing(
        desired_power_kw=desired_power_kw,
        desired_energy_kwh=desired_energy_kwh,
        desired_bank_voltage_v=desired_bank_voltage_v,
        cells_in_series=n_series,
        strings_in_parallel=n_parallel,
        total_cells=total_cells,
        nominal_bank_voltage_v=v_bank_nom,
        nominal_bank_capacity_kwh=bank_capacity_kwh,
        nominal_bank_capacity_ah=bank_capacity_ah,
        max_discharge_current_a=i_max_dis,
        max_charge_current_a=i_max_chg,
        max_c_rate_discharge_per_h=c_rate_dis,
        max_c_rate_charge_per_h=c_rate_chg,
        time_at_max_power_h=time_h,
    )


@dataclass
class BatteryBankDispatchModel:
    """
    Minimal dispatch model for MILP/simulation:
      - SOC state
      - power limits (kW)
      - current/C-rate limits derived from sizing
      - round-trip efficiency split into charge/discharge legs
    Convention:
      power_kw > 0  -> discharge to DC bus (battery output)
      power_kw < 0  -> charge from DC bus (battery input)
    """
    sizing: BatteryBankSizing
    soc: float = 0.50  # 0..1
    soc_min: float = 0.10
    soc_max: float = 0.90
    eta_dis: float = 0.96
    eta_chg: float = 0.96

    def step(self, requested_power_kw: float, dt_h: float) -> dict:
        """
        Apply a power request for duration dt_h, enforce limits, and update SOC.
        Returns a dict with delivered_power_kw (after limits), soc, and energies.
        """

        # 1) Power rating limit (as in your example: 300 kW DC)
        p_lim = self.sizing.desired_power_kw
        p_kw = max(-p_lim, min(p_lim, requested_power_kw))

        # 2) Current/C-rate limit -> translate Imax to Pmax at nominal V
        v = self.sizing.nominal_bank_voltage_v
        pmax_dis_kw = (self.sizing.max_discharge_current_a * v) / 1000.0
        pmax_chg_kw = (self.sizing.max_charge_current_a * v) / 1000.0

        if p_kw >= 0:
            p_kw = min(p_kw, pmax_dis_kw)
        else:
            p_kw = max(p_kw, -pmax_chg_kw)

        # 3) SOC limit -> compute max energy we can move this step
        e_cap = self.sizing.nominal_bank_capacity_kwh

        if p_kw >= 0:
            # discharging: battery energy decreases more than delivered due to efficiency
            e_out_dc = p_kw * dt_h
            e_from_batt = e_out_dc / max(self.eta_dis, 1e-9)

            e_available = (self.soc - self.soc_min) * e_cap
            if e_from_batt > e_available:
                e_from_batt = max(e_available, 0.0)
                e_out_dc = e_from_batt * self.eta_dis
                p_kw = e_out_dc / dt_h if dt_h > 0 else 0.0

            self.soc -= e_from_batt / e_cap if e_cap > 0 else 0.0
            self.soc = max(self.soc_min, self.soc)

            return {
                "mode": "discharge",
                "requested_power_kw": requested_power_kw,
                "delivered_power_kw": p_kw,
                "energy_to_dc_kwh": e_out_dc,
                "energy_from_battery_kwh": e_from_batt,
                "soc": self.soc,
            }

        else:
            # charging: battery energy increases less than absorbed due to efficiency
            e_in_dc = (-p_kw) * dt_h
            e_to_batt = e_in_dc * self.eta_chg

            e_room = (self.soc_max - self.soc) * e_cap
            if e_to_batt > e_room:
                e_to_batt = max(e_room, 0.0)
                e_in_dc = e_to_batt / max(self.eta_chg, 1e-9)
                p_kw = -(e_in_dc / dt_h) if dt_h > 0 else 0.0

            self.soc += e_to_batt / e_cap if e_cap > 0 else 0.0
            self.soc = min(self.soc_max, self.soc)

            return {
                "mode": "charge",
                "requested_power_kw": requested_power_kw,
                "delivered_power_kw": p_kw,  # negative
                "energy_from_dc_kwh": e_in_dc,
                "energy_to_battery_kwh": e_to_batt,
                "soc": self.soc,
            }


if __name__ == "__main__":
    # --- Use the exact example from your screenshot ---
    cell = BatteryCell(v_nom=3.6, capacity_ah=2.25)

    sizing = size_battery_bank(
        desired_power_kw=300.0,
        desired_energy_kwh=1200.0,
        desired_bank_voltage_v=500.0,
        cell=cell,
        # leaving C-rate None -> implied by 300/1200 = 0.25C (4-hour battery)
        max_c_rate_charge_per_h=None,
        max_c_rate_discharge_per_h=None,
    )

    print("=== Sizing (should match SAM closely) ===")
    for k, v in asdict(sizing).items():
        print(f"{k}: {v}")

    # --- Simple dispatch demo: discharge 300 kW for 1 hour from SOC=50% ---
    model = BatteryBankDispatchModel(sizing=sizing, soc=0.50)
    out = model.step(requested_power_kw=300.0, dt_h=1.0)
    print("\n=== Dispatch step ===")
    for k, v in out.items():
        print(f"{k}: {v}")


=== Sizing (should match SAM closely) ===
desired_power_kw: 300.0
desired_energy_kwh: 1200.0
desired_bank_voltage_v: 500.0
cells_in_series: 139
strings_in_parallel: 1066
total_cells: 148174
nominal_bank_voltage_v: 500.40000000000003
nominal_bank_capacity_kwh: 1200.2094000000002
nominal_bank_capacity_ah: 2398.5
max_discharge_current_a: 599.5203836930455
max_charge_current_a: 599.5203836930455
max_c_rate_discharge_per_h: 0.2499563826112343
max_c_rate_charge_per_h: 0.2499563826112343
time_at_max_power_h: 4.000698000000001

=== Dispatch step ===
mode: discharge
requested_power_kw: 300.0
delivered_power_kw: 299.99999999999994
energy_to_dc_kwh: 299.99999999999994
energy_from_battery_kwh: 312.49999999999994
soc: 0.23962876811329764
